# Results analysis

Before/after metrics and learning curves for a run. Set `RUN` to the run you want to look at.

Needs Jupyter (`pip install notebook`) and the outputs of `src.utils.compare` and `src.train`.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO))

RUN = 'ppo_carracing'
results_dir = REPO / 'results' / RUN
logs_dir = REPO / 'logs' / RUN

## Before vs. after

Raw CarRacing reward on the same evaluation tracks. `after` appears once a model has been trained and `compare --stage after` has run.

In [ ]:
metrics = {}
for stage in ('before', 'after'):
    path = results_dir / f'metrics_{stage}.json'
    if path.exists():
        metrics[stage] = json.loads(path.read_text(encoding='utf-8'))

keys = ['policy', 'n_episodes', 'mean_return', 'std_return', 'median_return', 'min_return', 'max_return', 'mean_length']
pd.DataFrame({stage: {k: m[k] for k in keys} for stage, m in metrics.items()})

In [ ]:
returns = pd.DataFrame({stage: pd.Series(m['returns']) for stage, m in metrics.items()})
ax = returns.plot(kind='box', title=f'{RUN}: episode returns', ylabel='Episode return (raw reward)')

## Learning curve

Needs the training logs in `logs/<run>/`.

In [ ]:
from src.utils.plotting import plot_learning_curve

if (logs_dir / 'monitor').exists():
    out = plot_learning_curve(logs_dir, results_dir / 'learning_curve.png', RUN)
    from IPython.display import Image, display
    display(Image(filename=str(out)))
else:
    print(f'No training logs in {logs_dir} yet.')